# Regime Parallel-Coordinates Plots (L5L6 — isothermal)

Interactive parallel-coordinates views of the **Level 5L6** regime-stratified sensitivity
analysis, built **entirely from the saved CSVs** in `sa_results/` — no model re-runs.

The L5L6 study is now run **isothermally** at three temperatures, so every table carries a
`temperature_K` column. Views A/B/D are drawn at one temperature at a time: mixing them would
put three pinned values of a non-sampled parameter on the same axes and pool three different
studies' clusters. View E uses temperature as the axis instead.

Views:

- **A** — samples coloured by **regime** (parameter space; the regime manifold)
- **B** — one plot **per regime**, coloured by **log10(flux)** (within-regime structure)
- **C** — **δ/floor sensitivity shift** across regimes (each line = a parameter)
- **D** — **output space** coloured by regime
- **E** — **δ/floor shift across temperature**, per regime

Views C and E plot **`delta_over_floor`**, not raw δ. The dummy-parameter noise floor differs
between clusters even at equal n, so raw δ is not comparable across the axes a PCP puts side by
side; the ratio is. **A value at or below 1.0 is not resolved** — indistinguishable from a
random input the model never saw.

> **The regime label ignores the parallel path.** L5L6 labels by argmax of the three *series*
> fractions (`frac_surface`, `frac_oxide`, `frac_metal`). The defect bypass is computed and
> `flux_intact`/`flux_defect` are recorded, but it never competes for the label — unlike the L5
> study, where `defect` is a peer regime. Worth remembering here, because `f_pinhole` turns out
> to be the strongest resolved driver of the *surface* cluster.

Interactive: drag on any axis to **brush/filter** the lines. A standalone HTML copy of each
figure is written to `sa_results/figures/`. Run with the **`mace_env`** kernel.

In [1]:
import os, sys, importlib
import numpy as np
import pandas as pd

parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

# This notebook must run with its OWN folder as the working directory: that is how
# calculations.config.model_config finds this folder's study_config.py, and how
# RESULTS_DIR below resolves. Fails loudly rather than silently using another study.
from calculations.config import assert_study_is_local
print(assert_study_is_local())

import calculations.sensitivity as sens
importlib.reload(sens)
from calculations.sensitivity import (
    parallel_coordinates_samples, parallel_coordinates_sensitivity, top_drivers,
    REGIME_LABELS,
)

# Figures from this notebook go in one folder beside it.
from calculations.figures import results_dir, save_figure
FIGS = results_dir("regime_parallel_coords")

study: fuerst_etal_2024_model_config  (Fuerst_et_al_2024/study_config.py, via folder)


study: fuerst_etal_2024_model_config  (Fuerst_et_al_2024/study_config.py, via folder)


In [2]:
RESULTS_DIR = "sa_results"
FIG_DIR     = results_dir("regime_parallel_coords")

# Which temperature to draw the per-sample views at. The L5L6 run is isothermal at
# 773 / 1073 / 1273 K; temperature is PINNED, not sampled, so it must not become an axis.
T_SHOW = 1273

master = pd.read_csv(os.path.join(RESULTS_DIR, "master_clusters.csv"))
routeB = pd.read_csv(os.path.join(RESULTS_DIR, "routeB_givendata.csv"))

TEMPS   = sorted(master['temperature_K'].unique())
REGIMES = list(REGIME_LABELS)                         # ('surface', 'oxide', 'metal')

# Slice to one temperature for the per-sample views.
master_T = master[master['temperature_K'] == T_SHOW].reset_index(drop=True)
routeB_T = routeB[routeB['temperature_K'] == T_SHOW].reset_index(drop=True)

# Columns shown on a log10 axis (span orders of magnitude).
LOG_COLS = {'P_upstream', 'k_diss_ref', 'k_diss_metal_ref', 'K_eq_ref', 'K_eq_metal_ref',
            'oxide_thickness', 'metal_thickness', 'D_ref', 'D_ox_ref', 'K_s_ref', 'K_ox_ref',
            'grain_size', 'gb_thickness', 'lattice_density',
            'trap_dislocation_N_T', 'trap_gb_N_T', 'trap_vacancy_N_T', 'trap_carbide_N_T',
            'f_pinhole', 'f_crack', 'f_gb_defect', 'flux', 'permeability', 'D_eff'}
def logsub(dims):
    return [d for d in dims if d in LOG_COLS]

print("temperatures:", TEMPS, "| drawing per-sample views at", T_SHOW, "K")
print("master rows:", len(master), f"(at {T_SHOW} K: {len(master_T)})")
print("clusters:", master_T['regime'].value_counts().to_dict())

temperatures: [773.0, 1073.0, 1273.0] | drawing per-sample views at 1273 K
master rows: 15434 (at 1273 K: 5085)
clusters: {'metal': 1729, 'oxide': 1685, 'surface': 1671}


## A — samples coloured by regime

Axes = union of each regime's top-5 δ drivers of flux (+ `log10(flux)`). Each line is one model
run; colour = regime. Brush an axis (e.g. pull `log10(P_upstream)` to its low end) to see which
regime/flux lines survive.

In [3]:
drivers = []
for r in REGIMES:
    for p in top_drivers(routeB_T, r, 'flux', k=5):
        if p not in drivers:
            drivers.append(p)
dimsA = drivers + ['flux']

figA = parallel_coordinates_samples(
    master_T, dimsA, color_by='regime', log_dims=logsub(dimsA),
    title=f'A — L5L6 samples by regime @ {T_SHOW} K (top δ drivers)',
    save_html=os.path.join(FIG_DIR, 'pcp_A_regime.html'))
print("A axes:", dimsA)
figA

A axes: ['f_pinhole', 'P_upstream', 'k_diss_metal_ref', 'E_diss', 'E_D_ox', 'H_sol_ox', 'K_s_ref', 'H_eq', 'trap_vacancy_N_T', 'D_ref', 'D_ox_ref', 'flux']


## B — per-regime clusters coloured by log10(flux)

One PCP per regime, axes = that regime's top-6 δ drivers, colour = `log10(flux)`. Clean
within-regime view (no cross-preset mixing).

In [4]:
figsB = {}
for r in REGIMES:
    dims = top_drivers(routeB_T, r, 'flux', k=6)
    sub  = master_T[master_T['regime'] == r]
    figsB[r] = parallel_coordinates_samples(
        sub, dims, color_by='flux', log_dims=logsub(dims),
        title=f'B — L5L6 {r} cluster @ {T_SHOW} K, coloured by log10(flux)',
        save_html=os.path.join(FIG_DIR, f'pcp_B_{r}.html'))
    print(f"{r:8s}: n={len(sub)}  axes={dims}")

for r in REGIMES:
    figsB[r].show()

surface : n=1671  axes=['f_pinhole', 'P_upstream', 'k_diss_metal_ref', 'E_diss', 'E_D_ox', 'lattice_density']
oxide   : n=1685  axes=['H_sol_ox', 'f_pinhole', 'K_s_ref', 'H_eq', 'trap_vacancy_N_T', 'D_ref']
metal   : n=1729  axes=['H_sol_ox', 'K_s_ref', 'D_ref', 'P_upstream', 'D_ox_ref', 'crack_thickness_factor']


## C — δ/floor sensitivity shift across regimes

Each line is a **parameter**; axes are the three regimes; height = its **`delta_over_floor`**
for log10(flux). Steeply sloped lines are parameters whose importance changes between regimes.
Colour = mean importance.

Read the 1.0 level as the noise floor: a parameter near or below it in every column is not
resolved anywhere, and its slope is noise rather than physics.

In [5]:
matC = (routeB_T[routeB_T.metric == 'flux']
        .pivot(index='parameter', columns='regime', values='delta_over_floor')
        .reset_index())
matC = matC[['parameter'] + [r for r in REGIMES if r in matC.columns]]

figC = parallel_coordinates_sensitivity(
    matC, regimes=tuple(REGIMES),
    title=f'C — L5L6 δ/floor shift across regimes @ {T_SHOW} K (log10 flux)',
    save_html=os.path.join(FIG_DIR, 'pcp_C_sensitivity.html'))
print("floors at this temperature:")
print(routeB_T[routeB_T.metric == 'flux'].groupby('regime')['floor'].first().round(4).to_string())
figC

floors at this temperature:
regime
metal      0.0913
oxide      0.1021
surface    0.0936


## D — output space coloured by regime

Axes = the model outputs; colour = regime. Shows how the regimes separate in *output* space —
the surface cluster collapsing to near-zero `theta` is the clearest separation.

In [6]:
dimsD = ['flux', 'theta', 'frac_surface', 'frac_oxide', 'frac_metal', 'permeability']
figD = parallel_coordinates_samples(
    master_T, dimsD, color_by='regime', log_dims=logsub(dimsD),
    title=f'D — L5L6 output space by regime @ {T_SHOW} K',
    save_html=os.path.join(FIG_DIR, 'pcp_D_outputs.html'))
figD

## E — δ/floor shift across temperature

One plot per regime. Each line is a **parameter**; axes are the three temperatures; height =
`delta_over_floor` for log10(flux).

A line rising steeply left to right is a parameter that only becomes resolvable at high
temperature. Lines hugging 1.0 across all three are noise at every temperature.

In [7]:
figsE = {}
for r in REGIMES:
    sub  = routeB[(routeB.metric == 'flux') & (routeB.regime == r)]
    matE = sub.pivot(index='parameter', columns='temperature_K', values='delta_over_floor')
    matE.columns = [f'{int(c)}K' for c in matE.columns]
    matE = matE.reset_index()
    tcols = tuple(f'{int(t)}K' for t in TEMPS)
    figsE[r] = parallel_coordinates_sensitivity(
        matE, regimes=tcols,
        title=f'E — L5L6 {r} regime: δ/floor vs temperature (log10 flux)',
        save_html=os.path.join(FIG_DIR, f'pcp_E_vs_T_{r}.html'))
    best = matE.set_index('parameter')[list(tcols)].max(axis=1).nlargest(3)
    print(f"{r:8s} top-3 by best δ/floor: {best.round(2).to_dict()}")

for r in REGIMES:
    figsE[r].show()

surface  top-3 by best δ/floor: {'P_upstream': 2.49, 'f_pinhole': 2.36, 'k_diss_metal_ref': 1.96}
oxide    top-3 by best δ/floor: {'H_sol_ox': 5.03, 'oxide_thickness': 2.38, 'K_s_ref': 1.47}
metal    top-3 by best δ/floor: {'H_sol_ox': 2.62, 'metal_thickness': 2.05, 'D_ref': 1.7}


In [8]:
# theta is the metric temperature does NOT dominate — worth its own cross-regime view
matT = (routeB_T[routeB_T.metric == 'theta']
        .pivot(index='parameter', columns='regime', values='delta_over_floor')
        .reset_index())
matT = matT[['parameter'] + [r for r in REGIMES if r in matT.columns]]
figT = parallel_coordinates_sensitivity(
    matT, regimes=tuple(REGIMES),
    title=f'C2 — L5L6 δ/floor across regimes, theta @ {T_SHOW} K',
    save_html=os.path.join(FIG_DIR, 'pcp_C2_theta.html'))
print("theta floors:")
print(routeB_T[routeB_T.metric == 'theta'].groupby('regime')['floor'].first().round(4).to_string())
figT

theta floors:
regime
metal      0.1229
oxide      0.1298
surface    0.1305


In [9]:
import glob
print("HTML figures written to", os.path.abspath(FIG_DIR), ":")
for f in sorted(glob.glob(os.path.join(FIG_DIR, '*.html'))):
    print("  ", os.path.basename(f), f"({os.path.getsize(f)//1024} KB)")

HTML figures written to /Users/akinyemi.az/Desktop/HE_papers/yoshi/Analytical_model/MHI_permeation/Fuerst_et_al_2024/regime_parallel_coords-results :
   pcp_A_regime.html (5486 KB)
   pcp_B_metal.html (4868 KB)
   pcp_B_oxide.html (4861 KB)
   pcp_B_surface.html (4863 KB)
   pcp_C2_theta.html (4728 KB)
   pcp_C_sensitivity.html (4728 KB)
   pcp_D_outputs.html (5160 KB)
   pcp_E_vs_T_metal.html (4728 KB)
   pcp_E_vs_T_oxide.html (4728 KB)
   pcp_E_vs_T_surface.html (4728 KB)
